In [4]:
import pandas as pd
from lifelines import CoxPHFitter

df = pd.read_csv("../data/cox/heart_failure_clinical_records_dataset.csv")

cph = CoxPHFitter()
cph.fit(df, duration_col='time', event_col='DEATH_EVENT')

betas = cph.params_.values           # pandas Series indexed by covariate name

print("\nBetas (array):")
print(betas)


Betas (array):
[ 4.64081786e-02  4.60135071e-01  2.20737247e-04  1.39884308e-01
 -4.89416023e-02  4.75749227e-01 -4.63520855e-07  3.21033841e-01
 -4.41874485e-02 -2.37521782e-01  1.28922483e-01]


In [5]:
import math

def chebyshev_nodes(n, interval):
    a, b = interval
    x, y = 0.5 * (a + b), 0.5 * (b - a)
    return [(x + y * math.cos((k - 0.5) * (math.pi / n)))
            for k in range(1, n + 1)]

def chebyshev_coeffs(op, nodes, interval):
    a, b = interval
    n = len(nodes)
    coeffs = [0.0 for _ in range(n)]
    y = [op(node) for node in nodes]

    for i in range(n):
        u = (2 * nodes[i] - a - b) / (b - a)
        t_prev = 1.0
        t = u

        for j in range(n):
            coeffs[j] += y[i] * t_prev
            t_next = 2 * u * t - t_prev
            t_prev = t
            t = t_next

    coeffs[0] /= n
    for i in range(1, n):
        coeffs[i] *= (2.0 / n)

    return coeffs

def chebyshev_evaluate_clenshaw(scaled_x, coeffs):
    n = len(coeffs)
    
    w_2 = scaled_x * (coeffs[n - 1] * 2) + coeffs[n - 2]
    w_1 = scaled_x * w_2 * 2 - (coeffs[n - 1] - coeffs[n - 3])
    w_0 = scaled_x * w_1 * 2 - w_2 + coeffs[n - 4]

    for i in range(n - 5, -1, -1):
        w_2, w_1 = w_1, w_0
        w_0 = scaled_x * w_1 * 2 - w_2 + coeffs[i]
    
    return w_0 - scaled_x * w_1

def via_chebyshev(x, op, interval, degree):
    coeffs = chebyshev_coeffs(
        op=op,
        nodes=chebyshev_nodes(degree + 1, interval),
        interval=interval)
    
    a, b = interval
    scaled_x = x * (2 / (b - a)) - (a + b) / (b - a)

    return chebyshev_evaluate_clenshaw(scaled_x, coeffs)

def approx_exp(value, interval, degree):
    return via_chebyshev(
            value, math.exp, interval, degree=degree)

In [6]:
test_cheby = approx_exp(np.array([1, 2, 3]), (0, 3), 20)

In [7]:
test_cheby

array([ 2.71828183,  7.3890561 , 20.08553692])

In [8]:
np.exp(np.array([1, 2, 3]))  # for comparison

array([ 2.71828183,  7.3890561 , 20.08553692])

In [12]:
import numpy as np
import pandas as pd

DEGREE = 10

def cox_ph_fit(X, T, E, tol=1e-6, maxiter=100, verbose=False, use_approx=True):
    """
    Fit Cox PH by maximizing the partial log-likelihood (Breslow ties handling).
    X: (n, p) covariate matrix (no intercept)
    T: (n,) event / censoring times
    E: (n,) event indicator (1=event, 0=censored)
    Returns: beta (p,), var_cov (p,p)
    """
    n, p = X.shape

    # sort by time descending so risk sets are prefixes
    order = np.argsort(-T)
    Xs = X[order]
    Es = E[order].astype(float)
    Ts = T[order]  # not used beyond ordering

    def _ll_grad_hess(beta, use_approx):
        eta = Xs.dot(beta)
        # for numerical stability shift
        eta -= np.max(eta)
        
        if use_approx:
            w = approx_exp(eta, interval=(-30, 0), degree=DEGREE)  # (n,)
        else:
            w = np.exp(eta)  # (n,)
        
        wX = w[:, None] * Xs  # (n,p)
        # wXX: (n,p,p)
        wXX = (Xs[:, :, None] * Xs[:, None, :]) * w[:, None, None]

        S0 = np.cumsum(w)              # (n,)
        S1 = np.cumsum(wX, axis=0)     # (n,p)
        S2 = np.cumsum(wXX, axis=0)    # (n,p,p)

        # contributions only where Es == 1
        xi_beta = (Xs * Es[:, None]).sum(axis=0)  # sum over events of x_i * 1
        # log-likelihood
        logS0 = np.log(S0 + 1e-16)
        ll = (Es * (Xs.dot(beta) - logS0)).sum()

        # gradient
        weighted_mean = S1 / S0[:, None]  # (n,p)
        grad = (Es[:, None] * (Xs - weighted_mean)).sum(axis=0)  # (p,)

        # hessian
        H = np.zeros((p, p))
        for i in range(n):
            if Es[i] == 0:
                continue
            S0_i = S0[i]
            S1_i = S1[i]
            S2_i = S2[i]
            mean = S1_i / S0_i
            H += (S2_i / S0_i) - np.outer(mean, mean)
        H = -H  # Hessian of log-likelihood is negative definite; for Newton we use H

        return ll, grad, H

    beta = np.zeros(p)
    for it in range(maxiter):
        ll, grad, H = _ll_grad_hess(beta, use_approx=use_approx)
        
        # regularize Hessian slightly for numerical stability
        try:
            step = np.linalg.solve(H, grad)
        except np.linalg.LinAlgError:
            H_reg = H - 1e-6 * np.eye(p)
            step = np.linalg.solve(H_reg, grad)
        
        # H_reg = H - 1e-6 * np.eye(p)
        # step = np.linalg.pinv(H_reg).dot(grad)
        
        # step = grad * 1e-10  # gradient ascent with small fixed lr
        
        beta_new = beta - step
        if verbose:
            print(f"iter {it}: ll={ll:.6f}, ||step||={np.linalg.norm(step):.6g}")
        if np.linalg.norm(step) < tol:
            beta = beta_new
            break
        beta = beta_new

    # compute final Hessian to get variance-cov
    _, _, H_final = _ll_grad_hess(beta, use_approx=use_approx)
    # var-cov approx = inverse(-H_final)
    try:
        var_cov = np.linalg.inv(-H_final)
    except np.linalg.LinAlgError:
        var_cov = np.linalg.pinv(-H_final)

    return beta, var_cov

# Use dataframe `df` already present in the notebook
# covariates are all columns except time and DEATH_EVENT
covariate_cols = [c for c in df.columns if c not in ("time", "DEATH_EVENT")]
X = df[covariate_cols].values
T = df["time"].values
E = df["DEATH_EVENT"].values

beta_est_approx, var_cov_approx = cox_ph_fit(X, T, E, verbose=True, use_approx=True)
beta_est_exact, var_cov_exact = cox_ph_fit(X, T, E, verbose=True, use_approx=False)

# present results in a pandas Series similar to lifelines
beta_series_approx = pd.Series(beta_est_approx, index=covariate_cols)
print("Estimated betas (numpy array, approx):")
print(beta_est_approx)
print("\nEstimated betas (Series, approx):")
print(beta_series_approx)

beta_series_exact = pd.Series(beta_est_exact, index=covariate_cols)
print("Estimated betas (numpy array, exact):")
print(beta_est_exact)
print("\nEstimated betas (Series, exact):")
print(beta_series_exact)

def concordance_index(T, E, scores):
    """Harrell's C-index (O(n^2) implementation). Higher score -> better concordance."""
    n = len(T)
    assert len(E) == n and len(scores) == n
    num = 0.0
    den = 0.0
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            # consider pair (i,j) if i had an observed event before j
            if T[i] < T[j] and E[i] == 1:
                den += 1
                if scores[i] > scores[j]:
                    num += 1
                elif scores[i] == scores[j]:
                    num += 0.5
    return num / den if den > 0 else np.nan

# compute risk scores (higher = higher hazard) for our two fits
scores_approx = X.dot(beta_est_approx)
scores_exact = X.dot(beta_est_exact)

print("Concordance (our approx):", concordance_index(T, E, scores_approx))
print("Concordance (our exact):", concordance_index(T, E, scores_exact))

# if lifelines betas (variable name `betas`) is present, evaluate it too
try:
    # handle if betas is a Series or numpy array; align if it's a Series with same index
    if isinstance(betas, pd.Series):
        betas_arr = betas.reindex(covariate_cols).values
    else:
        betas_arr = np.asarray(betas)
    scores_lifelines = X.dot(betas_arr)
    print("Concordance (lifelines):", concordance_index(T, E, scores_lifelines))
except Exception:
    print("lifelines betas not available; skipping lifelines concordance.")

iter 0: ll=-508.077164, ||step||=0.933335
iter 1: ll=nan, ||step||=6.50726
iter 2: ll=nan, ||step||=764.021
iter 3: ll=-71590.378563, ||step||=15.4271
iter 4: ll=-72580.804676, ||step||=15.4283
iter 5: ll=-73567.520836, ||step||=15.4284
iter 6: ll=-74550.508382, ||step||=15.4274
iter 7: ll=-75529.764662, ||step||=15.4255
iter 8: ll=-76505.301038, ||step||=15.4227
iter 9: ll=-77477.141108, ||step||=15.4191
iter 10: ll=-78445.319097, ||step||=15.4148
iter 11: ll=-79409.878421, ||step||=15.4098
iter 12: ll=-80370.870411, ||step||=15.4042
iter 13: ll=-81328.353165, ||step||=15.398
iter 14: ll=-82282.390536, ||step||=15.3914
iter 15: ll=-83233.051228, ||step||=15.3842
iter 16: ll=-84180.408000, ||step||=15.3766
iter 17: ll=-85124.536951, ||step||=15.3686
iter 18: ll=-86065.516902, ||step||=15.3603
iter 19: ll=-87003.428839, ||step||=15.3516
iter 20: ll=-87938.355429, ||step||=15.3426
iter 21: ll=-88870.380591, ||step||=15.3333
iter 22: ll=-89799.589118, ||step||=15.3237
iter 23: ll=-90726.0

/tmp/ipykernel_1922569/377448742.py:43: RuntimeWarning: invalid value encountered in log
  logS0 = np.log(S0 + 1e-16)


In [10]:
def cox_ph_fit_gd(X, T, E, lr=1e-4, maxiter=5000, tol=1e-6, verbose=False, use_approx=True):
    """
    Fit Cox PH by maximizing the partial log-likelihood using gradient ascent with simple backtracking.
    X: (n, p) covariate matrix (no intercept)
    T: (n,) event / censoring times
    E: (n,) event indicator (1=event, 0=censored)
    Returns: beta (p,), var_cov (p,p)
    """
    n, p = X.shape

    # sort by time descending so risk sets are prefixes
    order = np.argsort(-T)
    Xs = X[order]
    Es = E[order].astype(float)

    def _ll_grad_hess(beta):
        eta = Xs.dot(beta)
        eta -= np.max(eta)

        if use_approx:
            w = approx_exp(eta, interval=(-30, 0), degree=DEGREE)
        else:
            w = np.exp(eta)

        wX = w[:, None] * Xs
        wXX = (Xs[:, :, None] * Xs[:, None, :]) * w[:, None, None]

        S0 = np.cumsum(w)              # (n,)
        S1 = np.cumsum(wX, axis=0)     # (n,p)
        S2 = np.cumsum(wXX, axis=0)    # (n,p,p)

        logS0 = np.log(S0 + 1e-16)
        ll = (Es * (Xs.dot(beta) - logS0)).sum()

        weighted_mean = S1 / S0[:, None]  # (n,p)
        grad = (Es[:, None] * (Xs - weighted_mean)).sum(axis=0)  # (p,)

        return ll, grad

    beta = np.zeros(p)
    ll, grad = _ll_grad_hess(beta)

    for it in range(maxiter):
        step = lr * grad  # ascend
        beta_new = beta + step
        ll_new, grad_new = _ll_grad_hess(beta_new)

        # simple backtracking if objective decreased
        bt_tries = 0
        while ll_new < ll and bt_tries < 20:
            lr *= 0.5
            step = lr * grad
            beta_new = beta + step
            ll_new, grad_new = _ll_grad_hess(beta_new)
            bt_tries += 1

        if verbose and (it % 100 == 0 or bt_tries > 0):
            print(f"iter {it}: ll={ll_new:.6f}, lr={lr:.2e}, ||step||={np.linalg.norm(step):.6g}")

        if np.linalg.norm(step) < tol:
            beta = beta_new
            ll = ll_new
            break

        beta = beta_new
        ll = ll_new
        grad = grad_new

    return beta

# Fit using gradient ascent (approx and exact)
beta_gd_approx = cox_ph_fit_gd(X, T, E, lr=1e-4, maxiter=5000, tol=1e-6, verbose=True, use_approx=False)
beta_gd_exact  = cox_ph_fit_gd(X, T, E, lr=1e-4, maxiter=5000, tol=1e-6, verbose=True, use_approx=False)

# Present results as pandas Series
beta_series_gd_approx = pd.Series(beta_gd_approx, index=covariate_cols)
beta_series_gd_exact  = pd.Series(beta_gd_exact, index=covariate_cols)

print("Gradient-descent estimated betas (approx):")
print(beta_series_gd_approx)
print("\nGradient-descent estimated betas (exact):")
print(beta_series_gd_exact)

# Compare to lifelines and previous solutions if available
try:
    print("\nDifference (gd approx - lifelines):", (beta_series_gd_approx - betas).abs().sum())
    print("Difference (gd exact - lifelines):", (beta_series_gd_exact - betas).abs().sum())
    print("Difference (gd approx - our Newton exact):", (beta_series_gd_approx - beta_series_exact).abs().sum())
except Exception:
    pass

iter 0: ll=-1709.020043, lr=9.54e-11, ||step||=6.81374e-05
iter 100: ll=nan, lr=9.54e-11, ||step||=nan
iter 200: ll=nan, lr=9.54e-11, ||step||=nan
iter 300: ll=nan, lr=9.54e-11, ||step||=nan


iter 400: ll=nan, lr=9.54e-11, ||step||=nan
iter 500: ll=nan, lr=9.54e-11, ||step||=nan


/tmp/ipykernel_1922569/2732858496.py:35: RuntimeWarning: invalid value encountered in divide
  weighted_mean = S1 / S0[:, None]  # (n,p)


iter 600: ll=nan, lr=9.54e-11, ||step||=nan
iter 700: ll=nan, lr=9.54e-11, ||step||=nan
iter 800: ll=nan, lr=9.54e-11, ||step||=nan
iter 900: ll=nan, lr=9.54e-11, ||step||=nan
iter 1000: ll=nan, lr=9.54e-11, ||step||=nan
iter 1100: ll=nan, lr=9.54e-11, ||step||=nan
iter 1200: ll=nan, lr=9.54e-11, ||step||=nan
iter 1300: ll=nan, lr=9.54e-11, ||step||=nan
iter 1400: ll=nan, lr=9.54e-11, ||step||=nan
iter 1500: ll=nan, lr=9.54e-11, ||step||=nan
iter 1600: ll=nan, lr=9.54e-11, ||step||=nan
iter 1700: ll=nan, lr=9.54e-11, ||step||=nan
iter 1800: ll=nan, lr=9.54e-11, ||step||=nan
iter 1900: ll=nan, lr=9.54e-11, ||step||=nan
iter 2000: ll=nan, lr=9.54e-11, ||step||=nan
iter 2100: ll=nan, lr=9.54e-11, ||step||=nan
iter 2200: ll=nan, lr=9.54e-11, ||step||=nan
iter 2300: ll=nan, lr=9.54e-11, ||step||=nan
iter 2400: ll=nan, lr=9.54e-11, ||step||=nan
iter 2500: ll=nan, lr=9.54e-11, ||step||=nan
iter 2600: ll=nan, lr=9.54e-11, ||step||=nan
iter 2700: ll=nan, lr=9.54e-11, ||step||=nan
iter 2800: ll=